
# 1. Imports & APIs
Importing window functions and SQL modules required for statistical ranking (Quintiles).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# 2. Reference Date & Raw RFM
Calculating the dataset's 'current date' and aggregating the foundational Recency, Frequency, and Monetary metrics per unique human.

In [0]:
print("Calculating Base RFM Metrics...")

# Loading Silver tables (Single Source of Truth)
transactions = spark.table("workspace.silver_marketing_project.silver_marketing_transactions")
customers = spark.table("workspace.silver_marketing_project.silver_customer_profiles")

# Finding the dataset reference date (simulating 'today')
max_date = transactions.select(F.max("order_purchase_timestamp")).collect()[0][0]
ref_date = max_date + F.expr("INTERVAL 1 DAY")

# Calculating raw RFM metrics grouping by the TRUE customer identity
rfm_raw = transactions.join(customers, on="customer_id", how="inner") \
    .groupBy("customer_unique_id").agg(
        F.datediff(F.lit(ref_date), F.max("order_purchase_timestamp")).alias("recency_days"),
        F.count("order_id").alias("frequency_count"),
        F.sum("total_order_value").alias("monetary_sum")
    )


# 3. Statistical Scoring & Segmentation
Using quintiles to assign 1-5 scores to each metric and labeling customers with actionable Marketing Segments.

In [0]:
print("Applying Statistical Scoring and Business Logic...")

# Assigning 1-5 scores using NTILE (5 is best, 1 is worst)
window = Window.partitionBy()
rfm_scores = rfm_raw.select("*",
    F.ntile(5).over(window.orderBy(F.col("recency_days").desc())).alias("R_Score"),
    F.ntile(5).over(window.orderBy("frequency_count")).alias("F_Score"),
    F.ntile(5).over(window.orderBy("monetary_sum")).alias("M_Score")
)

# Creating the final marketing segments (Business Logic)
gold_rfm = rfm_scores.withColumn("Marketing_Segment", 
    F.when((F.col("R_Score") >= 4) & (F.col("F_Score") >= 4) & (F.col("M_Score") >= 4), "1. Champions")
     .when((F.col("R_Score") >= 3) & (F.col("F_Score") >= 3), "2. Loyal Customers")
     .when((F.col("R_Score") <= 2) & (F.col("F_Score") >= 4), "3. At Risk (Don't Lose Them)")
     .when((F.col("R_Score") <= 2) & (F.col("F_Score") <= 2), "5. Hibernating")
     .otherwise("4. Potential or Floating")
)

# Saving the final Executive Dashboard table
(gold_rfm.write
  .format("delta")
  .mode("overwrite")
  .option("comment", "EXECUTIVE GOLD TABLE: Customer RFM Scoring and Marketing Segmentation Ready for Power BI.")
  .saveAsTable("workspace.gold_marketing_project.gold_rfm_segments")
)
print("✓ Executive Gold Table created.")


# 4. Data Governance: Gold Columns
Documenting the core business metrics so C-Level executives understand the dashboard rules.

In [0]:
%sql
COMMENT ON COLUMN workspace.gold_marketing_project.gold_rfm_segments.customer_unique_id IS 'Unique customer identification';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_rfm_segments.recency_days IS 'Days since the last purchase (Lower is better)';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_rfm_segments.frequency_count IS 'Total number of distinct orders placed (Higher is better)';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_rfm_segments.monetary_sum IS 'Lifetime Value (LTV) - Total money spent in the store';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_rfm_segments.R_Score IS 'Statistical Quintile (1-5) for Recency';
COMMENT ON COLUMN workspace.gold_marketing_project.gold_rfm_segments.Marketing_Segment IS 'Final actionable label assigned for Marketing campaigns';